# Step 1: Install and import the required libraries

In your Jupyter Notebook, run this cell to install the necessary packages:

In [1]:
!pip install --upgrade google-api-python-client google-auth google-auth-oauthlib python-dotenv

  Using cached rsa-4.9-py3-none-any.whl.metadata (4.2 kB)
  Using cached oauthlib-3.2.2-py3-none-any.whl.metadata (7.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 45.1 MB/s eta 0:00:00 0:00:01
Using cached rsa-4.9-py3-none-any.whl (34 kB)
Using cached oauthlib-3.2.2-py3-none-any.whl (151 kB)
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.0.1
    Uninstalling python-dotenv-1.0.1:
      Successfully uninstalled python-dotenv-1.0.1


In [2]:
import os
from dotenv import load_dotenv
from googleapiclient.discovery import build
from google.oauth2 import service_account
import logging

# Step 2: Enable Google Sheets API and create Service Account

To connect your Python script to Google Sheets, you'll need to set up a project in Google Cloud Console, enable the Sheets API, and generate a service account key.

## 2.1 Create a project in Google Cloud Console
1. Go to: https://console.cloud.google.com/

2. Click on the top project selector and choose "New Project".

3. Give your project a name and create it.

## 2.2 Enable the Google Sheets API
1. With your project selected, go to this link:
https://console.cloud.google.com/apis/library/sheets.googleapis.com

2. Click the blue “Enable” button.

## 2.3 Create a Service Account
1. Go to: IAM & Admin > Service Accounts
https://console.cloud.google.com/iam-admin/serviceaccounts

2. Click "Create Service Account".

3. Give it a name like "sheets-bot" and click Create and Continue.

4. (You can skip the role selection for now — or choose "Editor" if needed.)

5. Click Done.

6. On the list of service accounts, find the one you just created and click on it.

7. Go to the “Keys” tab, then:

- Click "Add Key" > "Create New Key"
- Choose JSON format and download the file.

This JSON file contains your credentials — keep it safe.

## 2.4 Share the Google Sheet with your Service Account
- Open your Google Spreadsheet in the browser.

- Click "Share", and share it with the service account's email address.
It looks like: your-service-name@your-project-id.iam.gserviceaccount.com

Give it Editor access.





## 2.5 Create a .env file for secrets
In the same directory as your Jupyter Notebook, create a file called .env with this content:

```dotenv
SERVICE_ACCOUNT_FILE=./your-service-key.json
SPREADSHEET_ID=your_spreadsheet_id_here
```

Replace:

- your-service-key.json with the actual filename of your downloaded key.

- your_spreadsheet_id_here with the ID of your spreadsheet (the part between /d/ and /edit in its URL).

# Step 3: Authenticate and connect to Google Sheets
Now that you've set up your .env file and your service account, you can authenticate and access your spreadsheet.

Run this cell in your notebook:

In [3]:
# Load environment variables
load_dotenv()

# Get credentials and spreadsheet ID from environment
SERVICE_ACCOUNT_FILE = os.getenv('SERVICE_ACCOUNT_FILE')
SPREADSHEET_ID = os.getenv('SPREADSHEET_ID')

# Define the required API scope
SCOPES = ['https://www.googleapis.com/auth/spreadsheets']

# Authenticate and create a Google Sheets service object
creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)
service = build('sheets', 'v4', credentials=creds)
sheet = service.spreadsheets()

print("✅ Connected to Google Sheets successfully.")

✅ Connected to Google Sheets successfully.


# Step 4: Read and count rows from Sheet0

Let’s now read the data from your spreadsheet and count how many rows are currently in Sheet0. We'll skip the header row (assuming it's in row 1) and start reading from row 2 onward.

In [4]:
# Read data from Sheet0, starting from row 2 (to skip the header)
result = sheet.values().get(
    spreadsheetId=SPREADSHEET_ID,
    range='Sheet0!A2:Z'  # You can adjust this range depending on your actual columns
).execute()

rows = result.get('values', [])
num_rows = len(rows)

print(f"🔍 Number of rows (excluding header) in Sheet0: {num_rows}")


🔍 Number of rows (excluding header) in Sheet0: 4645


# Step 5: Remove duplicates from Sheet0 (based on column B)
In this step, we’ll go through each row in Sheet0, check the value in column B (which is index 1 in Python), and filter out any rows that have the same link. We’ll also log the number of duplicates found.

Add and run the following cell in your notebook:

In [5]:
# We'll track unique values in column B (index 1)
unique_links = set()
filtered_rows = []
duplicates = 0

for row in rows:
    # Make sure the row has at least 2 columns
    link = row[1] if len(row) > 1 else ''
    if link and link not in unique_links:
        unique_links.add(link)
        filtered_rows.append(row)
    else:
        duplicates += 1  # Count duplicates

print(f"✅ Finished filtering. {duplicates} duplicates found.")
print(f"🧹 Cleaned row count: {len(filtered_rows)}")


✅ Finished filtering. 3 duplicates found.
🧹 Cleaned row count: 4642


## Step 5.1: Clear old data and update Sheet0 with filtered rows
Run this cell in your notebook:

In [6]:
# First, clear the existing content in Sheet0 (excluding the header row)
clear_body = {}
clear_result = sheet.values().clear(
    spreadsheetId=SPREADSHEET_ID,
    range='Sheet0!A2:Z',  # Only clears rows below the header
    body=clear_body
).execute()

print("🧼 Old data (excluding header) cleared from Sheet0.")

🧼 Old data (excluding header) cleared from Sheet0.


Then, write the filtered (clean) data back to the sheet:

In [7]:
# Write the filtered rows back into Sheet0 starting from A2
update_body = {
    'values': filtered_rows
}

update_result = sheet.values().update(
    spreadsheetId=SPREADSHEET_ID,
    range='Sheet0!A2',
    valueInputOption='RAW',
    body=update_body
).execute()

print(f"✅ {len(filtered_rows)} rows written back to Sheet0 (without duplicates).")


✅ 4642 rows written back to Sheet0 (without duplicates).
